# cap3现金机会与SOXL选择逐段分解

只读保存的原生Qlib cap3和精确父账户，沿相同原价/分红/费用Qlib账户来源，944日/16季/819窗精确闭合。零新拟合、零新账户；区间由事后收益命名，不做交易规则。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
root=Path.cwd();card=root/"research/stage1_cap3_cash_selection_components_20261030"
if (card/"summary.json").exists():
    import qlib
    from src.core.recorder_sources import load_verified_recorder_objects
    summary=json.loads((card/"summary.json").read_text())
    assert summary["plan_sha256"]==hashlib.sha256((card/"plan.json").read_bytes()).hexdigest()
    qlib.init(provider_uri=str(root/"data/qlib/stage1_domain_heads_20261018"),region="us",kernels=1)
    saved,receipt=load_verified_recorder_objects(card,expected_recorder_id=summary["recorder_id"],
        artifacts=["analysis/daily","analysis/quarter","analysis/rolling126","analysis/summary"],
        expected_manifest_sha256=hashlib.sha256((card/"verification_manifest.json").read_bytes()).hexdigest())
    assert len(saved["analysis/daily"])==944 and len(saved["analysis/rolling126"])==819
    print("verified_existing_component_recorder",summary["recorder_id"])
else:
    import os
    os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
    os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
    import json,hashlib
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import qlib
    from qlib.workflow import R
    from src.core.recorder_sources import load_verified_recorder_objects
    from src.analysis.etf_transfer_attribution import holdings_attribution
    from src.workflow.stage1_nvda_guidance_receiver import prepare_account_sources
    root=Path.cwd();card=root/"research/stage1_cap3_cash_selection_components_20261030";plan_bytes=(card/"plan.json").read_bytes();plan=json.loads(plan_bytes)
    sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
    candidate=root/plan["candidate_source"]["card"];cs=plan["candidate_source"]
    assert sha(candidate/"summary.json")==cs["summary_sha256"] and sha(candidate/"verification_manifest.json")==cs["manifest_sha256"]
    parent=root/plan["parent_source"]["stage1_card"];ps=plan["parent_source"]
    assert sha(parent/"summary.json")==ps["stage1_summary_sha256"] and sha(parent/"verification_manifest.json")==ps["stage1_manifest_sha256"]
    assert sha(root/ps["provider_uri"]/"qlib_store_manifest.json")==ps["provider_manifest_sha256"]
    assert sha(root/plan["component_module"])==plan["component_module_sha256"]
    original=root/plan["original_component_source"]["component_card"];ospec=plan["original_component_source"]
    assert sha(original/"summary.json")==ospec["component_summary_sha256"] and sha(original/"verification_manifest.json")==ospec["manifest_sha256"]
    assert json.loads((original/"nas_binding.json").read_text())["evidence"]["sha256"]==ospec["component_NAS_archive_sha256"]
    helper=root/ps["account_helper_card"]
    assert sha(helper/"account_source_preflight.json")==ps["account_helper_receipt_sha256"]
    account=prepare_account_sources(root=root,card=helper)
    qlib.init(provider_uri=str(root/ps["provider_uri"]),region="us",kernels=1)
    cap,cap_receipt=load_verified_recorder_objects(candidate,expected_recorder_id=cs["recorder_id"],
        artifacts=["paths/no_beta_cap_0","analysis/rolling/all_windows"],expected_manifest_sha256=cs["manifest_sha256"])
    parent_objects,parent_receipt=load_verified_recorder_objects(parent,expected_recorder_id=ps["stage1_recorder_id"],
        artifacts=["inputs/frame"],expected_manifest_sha256=ps["stage1_manifest_sha256"])
    old_manifest=json.loads((original/"verification_manifest.json").read_text())
    old_entry=[e for e in old_manifest["entries"] if e["path"].endswith("/artifacts/analysis/daily")]
    assert len(old_entry)==1 and sha(original/old_entry[0]["path"])==old_entry[0]["sha256"]
    old_summary=json.loads((original/"summary.json").read_text())
    with R.uri_context(str(original/"qlib_recorder")):
        old_recorder=R.get_recorder(recorder_id=ospec["component_recorder_id"],experiment_id=old_summary["experiment_id"])
        old_daily=old_recorder.load_object("analysis/daily")
    assert sha(original/old_entry[0]["path"])==old_entry[0]["sha256"]
    old_receipt={"recorder_id":ospec["component_recorder_id"],"artifact":"analysis/daily","sha256":old_entry[0]["sha256"]}
    path=cap["paths/no_beta_cap_0"];frame=parent_objects["inputs/frame"].reset_index(drop=True);baseline=old_daily.reset_index(drop=True)
    assert len(frame)==len(path["replay"])==len(baseline)==plan["expected_sessions"]
    for name in ("signal_date","trade_date","exit_date"):
     assert pd.DatetimeIndex(pd.to_datetime(frame[name])).equals(pd.DatetimeIndex(pd.to_datetime(path["replay"][name])))
    book={"authority":path["replay"],"points":path["points"],"orders":path["orders"],"targets":path["targets"]}
    raw_targets=frame.set_index("signal_date")
    daily,segments=holdings_attribution(book,account["points"],account["schedule"],raw_targets,initial_cash=100000)
    assert len(daily)==944 and len(segments)==944*2*4 and daily.closure_error.abs().max()<1e-9
    ref=account["references"]["SOXX"]["authority"].reset_index(drop=True)
    assert pd.DatetimeIndex(ref.trade_date).equals(pd.DatetimeIndex(daily.trade_date))
    previous=np.r_[100000.,path["replay"].native_expected_nav.to_numpy(float)[:-1]]
    post=pd.DatetimeIndex(daily.trade_date)+pd.Timedelta(hours=9,minutes=47)
    marks=path["points"].set_index("stamp").loc[post];prices=account["points"]
    w_soxl=marks.q_soxl.to_numpy(float)*prices.loc[post,"SOXL"].to_numpy(float)/previous
    w_cash=(marks.cash.to_numpy(float)+marks.receivable.to_numpy(float))/previous
    candidate_return=daily["return"].to_numpy(float);benchmark=ref.strategy_return.to_numpy(float)
    cash=-w_cash*benchmark
    selection=daily.SOXL_contribution.to_numpy(float)-w_soxl*benchmark
    fees=daily.fee_contribution.to_numpy(float)
    relative=candidate_return-benchmark
    residual=relative-cash-selection-fees
    paired=np.log1p(candidate_return)-np.log1p(benchmark)
    scale=np.divide(paired,relative,out=1/(1+benchmark),where=np.abs(relative)>1e-12)
    comp=pd.DataFrame({"signal_date":frame.signal_date,"trade_date":daily.trade_date,"exit_date":daily.exit_date,
        "paired_log_excess":paired,"cash_log":cash*scale,"soxl_selection_log":selection*scale,
        "direct_fees_log":fees*scale,"timing_benchmark_residual_log":residual*scale,
        "post_open_cash_weight":w_cash,"post_open_soxl_weight":w_soxl,
        "raw_soxl_weight":frame.soxl_weight,"raw_soxx_weight":frame.soxx_weight,"raw_cash_weight":frame.cash_weight,
        "alpha_probability":frame.alpha_probability,"tail_probability":frame.tail_probability,
        "risk_state":daily.risk_state})
    fields=["cash_log","soxl_selection_log","direct_fees_log","timing_benchmark_residual_log"]
    assert np.max(np.abs(comp[fields].sum(axis=1)-comp.paired_log_excess))<1e-12
    assert abs(float(np.exp(comp.paired_log_excess.sum()))-json.loads((candidate/"summary.json").read_text())["SOXX_metrics"]["relative_wealth"])<1e-9
    assert pd.DatetimeIndex(comp.trade_date).equals(pd.DatetimeIndex(baseline.trade_date))
    baseline_rename={"cash_opportunity_log":"parent_cash_log","SOXL_selection_log":"parent_soxl_selection_log","Stage1_direct_fees_log":"parent_direct_fees_log","timing_and_benchmark_residual_log":"parent_timing_residual_log","paired_log_excess":"parent_paired_log_excess"}
    for col,name in baseline_rename.items():comp[name]=baseline[col].to_numpy(float)
    comp["year"]=pd.DatetimeIndex(comp.trade_date).year
    comp["quarter"]=pd.DatetimeIndex(comp.trade_date).to_period("Q").astype(str)
    
    def describe(group,kind,period):
     result={"kind":kind,"period":str(period),"sessions":len(group),"first_trade":str(pd.Timestamp(group.trade_date.iloc[0]).date()),"last_exit":str(pd.Timestamp(group.exit_date.iloc[-1]).date())}
     for name in ["paired_log_excess",*fields,*baseline_rename.values()]:result[name]=float(group[name].sum())
     result["relative_wealth_vs_soxx"]=float(np.exp(result["paired_log_excess"]))
     result["relative_wealth_vs_parent"]=float(np.exp(result["paired_log_excess"]-result["parent_paired_log_excess"]))
     for name in ("cash_log","soxl_selection_log","direct_fees_log"):
      result[name+"_vs_parent"]=float(result[name]-result[{"cash_log":"parent_cash_log","soxl_selection_log":"parent_soxl_selection_log","direct_fees_log":"parent_direct_fees_log"}[name]])
     result["mean_post_open_soxl_weight"]=float(group.post_open_soxl_weight.mean())
     result["mean_post_open_cash_weight"]=float(group.post_open_cash_weight.mean())
     result["mean_raw_soxl_weight"]=float(group.raw_soxl_weight.mean())
     result["mean_alpha_probability"]=float(group.alpha_probability.mean())
     result["mean_tail_probability"]=float(group.tail_probability.mean())
     return result
    annual=pd.DataFrame([describe(g,"year",y) for y,g in comp.groupby("year",sort=True)])
    quarter=pd.DataFrame([describe(g,"quarter",q) for q,g in comp.groupby("quarter",sort=True)])
    rolling=pd.DataFrame([describe(comp.iloc[i:i+126],"rolling126",i) for i in range(len(comp)-125)])
    assert len(quarter)==plan["expected_quarters"] and len(rolling)==plan["expected_rolling126"]
    saved=cap["analysis/rolling/all_windows"]
    assert len(saved)==len(rolling)
    assert np.max(np.abs(rolling.paired_log_excess.to_numpy(float)-saved.relative_log.to_numpy(float)))<1e-9
    quarter.to_csv(card/"quarter_components.csv",index=False)
    annual.to_csv(card/"annual_components.csv",index=False)
    focus=quarter.loc[quarter.period.isin(["2024Q3","2026Q2","2026Q3"])].to_dict("records")
    extremes={"worst":rolling.loc[rolling.paired_log_excess.idxmin()].to_dict(),"best":rolling.loc[rolling.paired_log_excess.idxmax()].to_dict()}
    (card/"window_extremes.json").write_text(json.dumps(extremes,ensure_ascii=False,indent=2)+"\n")
    tracking=card/"qlib_recorder"
    with R.start(experiment_name=card.name,uri=str(tracking)):
     rec=R.get_recorder();rec.save_objects(artifact_path="inputs",plan=plan,source_audits={"cap3":cap_receipt,"parent":parent_receipt,"original_components":old_receipt})
     rec.save_objects(artifact_path="sources",analysis_module=(root/plan["component_module"]).read_text())
     rec.save_objects(artifact_path="analysis",daily=comp,segments=segments,annual=annual,quarter=quarter,rolling126=rolling,window_extremes=extremes)
     rec.log_metrics(relative_wealth=float(np.exp(comp.paired_log_excess.sum())),max_component_closure_error=float(np.max(np.abs(comp[fields].sum(axis=1)-comp.paired_log_excess))))
    summary={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),"decision":"read_only_cap3_cash_selection_components_no_policy",
     "sessions":len(comp),"quarters":len(quarter),"rolling126_windows":len(rolling),"full_relative_wealth":float(np.exp(comp.paired_log_excess.sum())),
     "full_components":{k:float(comp[k].sum()) for k in [*fields,*baseline_rename.values()]},"focus":focus,"window_extremes":extremes,
     "new_model_fits":0,"new_native_accounts":0,"historical_research_passed":False,"recorder_id":rec.id,"experiment_id":str(rec.experiment_id),
     "recorder_uri":str(tracking),"recorder_local_dir":str(tracking/str(rec.experiment_id)/rec.id),
     "artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":["inputs/*","sources/*","analysis/*"]},
     "limitations":["Physical-path components are descriptive, not causal or tradable counterfactuals.","Cap3 source Recorder lacks NAS artifact receipt and must remain local.","2023-2026 history is reused; no new signal or account was fitted."]}
    (card/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n")
    print(json.dumps({"decision":summary["decision"],"full_components":summary["full_components"],"focus":focus},ensure_ascii=False))

## 中文结论

cap3同源原生持仓现金与SOXX基准在944日/16季/819窗逐日闭合。全程相对SOXX财富0.883；现金机会对数−0.51983、SOXL相对选择+0.39572、费用−0.00723、时点/基准残差+0.00691，仍不足跨过SOXX。原f696相应现金−0.60032、选择+0.22593，cap3两项虽改善但未取得稳定超额。2024Q3现金保护+0.0134和SOXL选择−0.06956，使cap3比原账户差；2026Q2选择+0.31708覆盖现金−0.13862；2026Q3现金−0.01502和选择−0.12424共同拖累，虽比原账户略好仍落后SOXX。最弱126日2023-05-15至11-13两项均负；最强2025-11-21至2026-05-27选择收益覆盖现金损失。全部是事后实际路径会计归因，不是阈值/因果反事实。零新拟合/账户，SOXX0bp已失败，2/5bp未过。新Qlib Recorder上传NAS并限定读回通过，cap3原件仍needs_nas。下一步必须找决策前同时区分现金机会与SOXL选择的来源信息，否则不拟合。